[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pachterlab/OP_2025_2/blob/scope-future/analysis_scripts/Large_Arrays_Demo.ipynb)

# accumap on large arrays: multilevel start, fold-free selection and relocation

This notebook demonstrates accumap's large-array path on a simulated shared-bead array with known bead positions:

1. **Large-array mode** (`count_model.LENGTH_REF_UM`): local constants (kernel range, warp grids, refinement windows) keep
   their physical size, so cost grows roughly linearly with the number of beads.
2. **Multilevel start** (`accumap.multilevel_layout`): beads are merged into super-beads level by level, the coarsest level is
   reconstructed (UMAP + accumap), and the layout is refined back up the levels.
3. **Fold-free selection for elongated arrays**: UMAP's default (spectral) initialization folds rectangles longer than about
   2:1. For elongated footprints the coarsest level also tries random initializations and keeps the one with the least
   density excess (a fold crowds beads).
4. **Relocation** (`accumap.relocate_passes`): beads left outside the footprint, or far from all their strongest partners, are
   moved next to their well-placed partners.

The demo array is a 3 x 7.2 mm rectangle (aspect 2.4, like SCOPE's 17 x 41 mm eye chart) with about 37,000 beads.

**Runtime**: use a GPU runtime (*Runtime > Change runtime type > GPU*). The whole notebook takes about 25-30 minutes (on a GPU
like Colab's T4 with two CPU cores: about 24 minutes in testing), mostly the two starts and accumap from the folded start.

In [ ]:
# Colab only: clone the repository (scope-future branch) and install accumap; skipped when run locally
import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("/content/OP_2025_2"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "scope-future", "https://github.com/pachterlab/OP_2025_2.git", "/content/OP_2025_2"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/OP_2025_2", "numba"], check=True)
if "google.colab" in sys.modules:
    os.chdir("/content/OP_2025_2/analysis_scripts")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import accumap
import accumap.count_model as cm
import accumap.multilevel as ml

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this notebook needs a GPU runtime)")

## Settings

`LENGTH_REF_UM` switches accumap to large-array mode: lengths are scaled so that an array of this diameter behaves as the
published method does; larger arrays keep the same physical kernel range. 7,000 um is the value used for SCOPE-like arrays.

The multilevel start's coarsest level is set smaller than the default (1,500 instead of 5,000 super-beads) and tries 6
random initializations instead of 16, to keep the demo short.

In [ ]:
W, H = 3000.0, 7200.0               # array size (um): a 2.4:1 rectangle
cm.LENGTH_REF_UM = 7000.0           # large-array mode
ml.COARSE_RANDOM_TRIES = 6          # random-init UMAPs at the coarsest level (default 16)
COARSEST_NODES = 1500               # super-beads at the coarsest level (default 5000)

footprint = np.array([[0, 0], [W, 0], [W, H], [0, H]], float) - [W / 2, H / 2]   # array outline (um), any polygon
diameter = float(np.hypot(W, H))

## Simulate an array with known bead positions

`simulate_shared_array` places beads on a jittered hexagonal lattice (26 um spacing) and draws counts between every pair
from a Poisson model with the transfer kernel and background fitted to SCOPE's real circular array (shared beads: every bead
both sends and receives).

In [ ]:
t0 = time.time()
X, C = accumap.simulate_shared_array(W, H, seed=0, verbose=True)
print(f"{C.shape[0]:,} beads, median {np.median(np.asarray(C.sum(1)).ravel() + np.asarray(C.sum(0)).ravel()):.0f} counts per bead "
      f"(sent + received), {time.time() - t0:.0f} s")

## Helpers: robust error and plots

Errors are measured after a similarity alignment (translation, rotation, reflection, one scale) to the true positions. The
alignment is refit on the best-placed 90% of beads, so a misplaced patch does not drag the rest. Layouts are shown coloured by
each bead's true position along the short and the long side: a correct layout shows two smooth, perpendicular gradients.

In [ ]:
def robust_error(Y, X, keep=0.9):
    """Per-bead error (um) after a similarity alignment refit on the best-placed `keep` fraction of beads."""
    Y, X = np.asarray(Y, float), np.asarray(X, float)
    A, T = accumap.similarity_align(Y, X)
    e = np.linalg.norm(A - T, axis=1)
    for _ in range(3):
        m = e <= np.quantile(e, keep)
        P, Q = Y[m] - Y[m].mean(0), X[m] - X[m].mean(0)
        U, s, Vt = np.linalg.svd(P.T @ Q)
        e = np.linalg.norm((Y - Y[m].mean(0)) @ (U @ Vt) * (s.sum() / np.sum(P ** 2)) - (X - X[m].mean(0)), axis=1)
    return e

results = {}
def score(name, Y):
    e = robust_error(Y, X)
    results[name] = {"robust median (um)": np.median(e), "p90 (um)": np.percentile(e, 90), "beads > 1 mm off (%)": 100 * (e > 1000).mean()}
    print(f"{name:28s} robust median {np.median(e):7.1f} um | p90 {np.percentile(e, 90):7.1f} um | > 1 mm off {100 * (e > 1000).mean():5.1f}%")
    return e

def show(Y, title, e=None):
    """Layout coloured by true short-side and long-side position, and (if given) by its error."""
    Y = np.asarray(Y, float)
    fig, ax = plt.subplots(1, 3 if e is not None else 2, figsize=(14 if e is not None else 9.5, 4.2))
    for k, (lab, col) in enumerate((("true x (short side)", X[:, 0]), ("true y (long side)", X[:, 1]))):
        ax[k].scatter(Y[:, 0], Y[:, 1], c=col, s=0.3, cmap="turbo"); ax[k].set_title(f"{title}\ncoloured by {lab}", fontsize=10)
    if e is not None:
        sc = ax[2].scatter(X[:, 0], X[:, 1], c=np.minimum(e, 1000), s=0.3, cmap="magma")
        ax[2].set_title("error at each bead's true position (um, capped at 1 mm)", fontsize=10); plt.colorbar(sc, ax=ax[2])
    for a in ax:
        a.set_aspect("equal"); a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()

## 1. Default start: UMAP, then accumap

The published starting layout is UMAP with its default spectral initialization (here the faster multithreaded settings).
On a rectangle longer than about 2:1, the first Laplacian eigenvectors are cos(pi y/L) and cos(2 pi y/L): both run along the long
side, so the initial layout folds the two halves onto each other. accumap's warp and refinement are local and cannot undo a
fold.

In [ ]:
t0 = time.time()
E0 = accumap.umap_layout(C, seed=0, shared=True, fast=True)
print(f"UMAP: {time.time() - t0:.0f} s")
e = score("UMAP start", E0); show(E0, "UMAP start (spectral init)", e)

In [ ]:
t0 = time.time()
R0 = accumap.reconstruct(C, E0, diameter, shared=True, footprint=footprint, verbose=False)
print(f"accumap: {time.time() - t0:.0f} s")
e = score("accumap from UMAP", R0["receivers"]); show(R0["receivers"], "accumap from the UMAP start", e)

## 2. Multilevel start with fold-free selection and relocation

`multilevel_layout` coarsens the counts to at most `target` super-beads, reconstructs the coarsest level, and refines back up.
Because the footprint is elongated (aspect >= 1.5), the coarsest level tries the default UMAP and `COARSE_RANDOM_TRIES` random
initializations, runs accumap on each, and keeps the layout with the least density excess. At the end, relocation passes move
misplaced beads next to their well-placed partners.

In [ ]:
t0 = time.time()
E1, info = accumap.multilevel_layout(C, footprint=footprint, seed=0, target=COARSEST_NODES)
print(f"multilevel start: {time.time() - t0:.0f} s, coarsest level {info['coarse_nodes']:,} super-beads")
e = score("multilevel start", E1); show(E1, "multilevel start", e)

In [ ]:
t0 = time.time()
R1 = accumap.reconstruct(C, E1, diameter, shared=True, footprint=footprint, verbose=True)
print(f"accumap: {time.time() - t0:.0f} s")
Y1, n_flagged = accumap.relocate_passes(C, np.asarray(R1["receivers"], float), footprint)   # relocation on accumap's result
e = score("accumap from multilevel", Y1); show(Y1, "accumap from the multilevel start (+ relocation)", e)

## Summary

In [ ]:
import pandas as pd
pd.DataFrame(results).T.round(1)

## Scaling up

The same calls handle much larger arrays; what changes is memory and time.

* **Memory**: in large-array mode the counts are stored compactly on the GPU (about 6 bytes per nonzero pair) when they fit,
  and otherwise streamed from pinned host memory; refinement partner lists are padded on the GPU when they fit.
* **Two GPUs**: set `cm.MULTI_GPU = ["cuda:0", "cuda:1"]` to split the warp's observed and expected terms and the refinement
  scoring across both. Each GPU keeps as much of its share of the counts resident as `cm.MULTI_GPU_RESERVE_GB` allows and
  streams the rest.
* **Distinct receivers and senders** (Hu et al.'s design): use `accumap.multilevel_layout_distinct(C, diameter)`.

Results on simulated arrays (two 8 GB GPUs; robust median error after trimmed alignment):

| Array | Beads | Nonzero pairs | Start + accumap | Robust median | Beads > 1 mm off |
|---|---|---|---|---|---|
| 18 mm square, shared | 553k | 443M | ~11 + 11 min | 7.5 um | 0% |
| 17.2 x 41 mm rectangle (eye-chart size), shared | 1.20M | 1.67B | ~38 + 52 min | 51 um | 0.07% |

On the eye-chart-sized array, the default start folded (79% of beads more than 1 mm off); the fold-free selection and
relocation shown here brought it to the numbers above.